In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)


TRAIN_PATH = "../data/processed/train_table_train_2015_2024.parquet"
TEST_PATH  = "../data/processed/train_table_test_2025.parquet"
OUT_DIR    = "../models"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42


train_table = pd.read_parquet(TRAIN_PATH)
test_table  = pd.read_parquet(TEST_PATH)
train_table["time_bin"] = pd.to_datetime(train_table["time_bin"])
test_table["time_bin"]  = pd.to_datetime(test_table["time_bin"])


train_df = train_table[train_table["time_bin"].dt.year <= 2023].copy()
val_df   = train_table[train_table["time_bin"].dt.year == 2024].copy()
test_df  = test_table.copy()

print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)
print("y mean:", train_df["y"].mean(), val_df["y"].mean(), test_df["y"].mean())


drop_cols = ["y", "time_bin"]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df["y"].astype(int)

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["y"].astype(int)

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["y"].astype(int)

cat_cols = ["cell_id"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    class_weight=None,  
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("rf", rf),
])


pipe.fit(X_train, y_train)


def eval_split(name, X, y):
    proba = pipe.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    ap  = average_precision_score(y, proba)
    print(f"\n[{name}] AUC={auc:.4f}  AP={ap:.4f}")
    print("Confusion:\n", confusion_matrix(y, pred))
    print(classification_report(y, pred, digits=4))

eval_split("VAL(2024)", X_val, y_val)
eval_split("TEST(2025)", X_test, y_test)


joblib.dump(pipe, os.path.join(OUT_DIR, "rf_grid.pkl"))
print("\nSaved:", os.path.join(OUT_DIR, "rf_grid.pkl"))

Train/Val/Test: (292810, 15) (33019, 15) (31773, 15)
y mean: 0.5859157815648373 0.6036221569399437 0.570264060680452

[VAL(2024)] AUC=0.8520  AP=0.8965
Confusion:
 [[ 8804  4284]
 [ 3081 16850]]
              precision    recall  f1-score   support

           0     0.7408    0.6727    0.7051     13088
           1     0.7973    0.8454    0.8207     19931

    accuracy                         0.7769     33019
   macro avg     0.7690    0.7590    0.7629     33019
weighted avg     0.7749    0.7769    0.7748     33019


[TEST(2025)] AUC=0.8507  AP=0.8815
Confusion:
 [[ 9389  4265]
 [ 2944 15175]]
              precision    recall  f1-score   support

           0     0.7613    0.6876    0.7226     13654
           1     0.7806    0.8375    0.8081     18119

    accuracy                         0.7731     31773
   macro avg     0.7709    0.7626    0.7653     31773
weighted avg     0.7723    0.7731    0.7713     31773


Saved: ../models/rf_grid.pkl
